# Experiment 4
## Generative Adversarial Network (GAN) on MNIST
**Aim:** Train a GAN on MNIST and monitor generator and discriminator losses.

### Step 1 – Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

### Step 2 – Load MNIST Data

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_data   = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
print('Dataset loaded:', len(train_data), 'images')

### Step 3 – Define Generator and Discriminator

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(100, 256), nn.ReLU(),
            nn.Linear(256, 512), nn.ReLU(),
            nn.Linear(512, 784), nn.Tanh())
    def forward(self, z): return self.net(z).view(-1, 1, 28, 28)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 1),   nn.Sigmoid())
    def forward(self, x): return self.net(x.view(-1, 784))

### Step 4 – Initialize Models, Loss & Optimizers

In [ ]:
G = Generator()
D = Discriminator()
criterion = nn.BCELoss()
opt_G = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))
print('Generator and Discriminator initialized.')

### Step 5 – Training Loop (5 Epochs)

In [ ]:
G_losses, D_losses = [], []
for epoch in range(5):
    g_loss_sum = d_loss_sum = 0
    for real_imgs, _ in train_loader:
        batch = real_imgs.size(0)
        real_labels = torch.ones(batch, 1)
        fake_labels = torch.zeros(batch, 1)

        # Train Discriminator
        z = torch.randn(batch, 100)
        fake_imgs = G(z).detach()
        d_loss = criterion(D(real_imgs), real_labels) + criterion(D(fake_imgs), fake_labels)
        opt_D.zero_grad(); d_loss.backward(); opt_D.step()

        # Train Generator
        z = torch.randn(batch, 100)
        g_loss = criterion(D(G(z)), real_labels)
        opt_G.zero_grad(); g_loss.backward(); opt_G.step()

        g_loss_sum += g_loss.item(); d_loss_sum += d_loss.item()
    G_losses.append(g_loss_sum/len(train_loader))
    D_losses.append(d_loss_sum/len(train_loader))
    print(f'Epoch {epoch+1}/5 | G Loss: {G_losses[-1]:.4f} | D Loss: {D_losses[-1]:.4f}')

### Step 6 – Plot Generator and Discriminator Losses

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, 6), G_losses, label='Generator Loss',     marker='o')
plt.plot(range(1, 6), D_losses, label='Discriminator Loss', marker='s')
plt.title('GAN Training Losses'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(True); plt.show()

### Step 7 – Visualize Generated Images

In [ ]:
z = torch.randn(16, 100)
fake_imgs = G(z).detach().squeeze()
fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(fake_imgs[i].numpy(), cmap='gray')
    ax.axis('off')
plt.suptitle('GAN Generated Images'); plt.tight_layout(); plt.show()

### Result
The GAN was trained on MNIST. Generator and Discriminator losses were monitored, and the generator learned to produce digit-like images.